# Figure: Full MIA Pipeline (Precision / Recall / AUC per $\kappa$)

Extended ML-Leaks pipeline: per-$\kappa$ panels of precision, recall, and AUC for both MLP and CNN target models. Provides the per-architecture detail behind the AUC summary in `04_mia_auc_curves`.

**External deps:** `classifier.py`, `mlLeaks.py` (not yet in this repo).


In [ ]:
%%writefile improved_cnn.py

"""
Improved CNN Architectures for CIFAR-10
This file contains enhanced CNN models to replace the simple CNNNet.
All models are Opacus-compatible (no in-place operations).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ImprovedCNN(nn.Module):
    """
    Modern CNN with:
    - Batch normalization for better training stability
    - Dropout for regularization
    - More convolutional layers for better feature extraction
    - Gradually increasing channels

    Architecture:
    Conv(32) -> BN -> ReLU -> Conv(32) -> BN -> ReLU -> MaxPool -> Dropout
    -> Conv(64) -> BN -> ReLU -> Conv(64) -> BN -> ReLU -> MaxPool -> Dropout
    -> Conv(128) -> BN -> ReLU -> MaxPool -> Dropout
    -> FC(256) -> ReLU -> Dropout -> FC(10)
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        _, C, H, W = n_in

        # First block: 32 channels
        self.conv1a = nn.Conv2d(C, 32, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(32)
        self.conv1b = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(0.2)

        # Second block: 64 channels
        self.conv2a = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(64)
        self.conv2b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(0.3)

        # Third block: 128 channels
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.drop3 = nn.Dropout(0.4)

        # Calculate flattened dimension
        with torch.no_grad():
            dummy = torch.zeros(1, C, H, W)
            x = self.pool1(F.relu(self.bn1b(self.conv1b(F.relu(self.bn1a(self.conv1a(dummy)))))))
            x = self.pool2(F.relu(self.bn2b(self.conv2b(F.relu(self.bn2a(self.conv2a(x)))))))
            x = self.pool3(F.relu(self.bn3(self.conv3(x))))
            flat_dim = x.numel()

        # Fully connected layers
        self.fc1 = nn.Linear(flat_dim, 256)
        self.drop_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, n_out)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1a(self.conv1a(x)), inplace=False)
        x = F.relu(self.bn1b(self.conv1b(x)), inplace=False)
        x = self.pool1(x)
        x = self.drop1(x)

        # Block 2
        x = F.relu(self.bn2a(self.conv2a(x)), inplace=False)
        x = F.relu(self.bn2b(self.conv2b(x)), inplace=False)
        x = self.pool2(x)
        x = self.drop2(x)

        # Block 3
        x = F.relu(self.bn3(self.conv3(x)), inplace=False)
        x = self.pool3(x)
        x = self.drop3(x)

        # Classifier
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x), inplace=False)
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x


class VGGStyleCNN(nn.Module):
    """
    VGG-inspired architecture optimized for CIFAR-10.

    Uses small 3x3 filters with multiple layers like VGG.
    More parameters and capacity than ImprovedCNN.

    Architecture pattern:
    [Conv-BN-ReLU] x 2 -> MaxPool -> Dropout
    [Conv-BN-ReLU] x 2 -> MaxPool -> Dropout
    [Conv-BN-ReLU] x 3 -> MaxPool -> Dropout
    FC -> ReLU -> Dropout -> FC
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        _, C, H, W = n_in

        # Block 1: 64 channels
        self.conv1_1 = nn.Conv2d(C, 64, kernel_size=3, padding=1)
        self.bn1_1 = nn.BatchNorm2d(64)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1_2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(0.2)

        # Block 2: 128 channels
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2_1 = nn.BatchNorm2d(128)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2_2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(0.3)

        # Block 3: 256 channels
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3_1 = nn.BatchNorm2d(256)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3_2 = nn.BatchNorm2d(256)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3_3 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(2)
        self.drop3 = nn.Dropout(0.4)

        # Calculate flattened dimension
        with torch.no_grad():
            dummy = torch.zeros(1, C, H, W)
            x = self._forward_conv_blocks(dummy)
            flat_dim = x.numel()

        # Classifier
        self.fc1 = nn.Linear(flat_dim, 512)
        self.drop_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, n_out)

    def _forward_conv_blocks(self, x):
        # Block 1
        x = F.relu(self.bn1_1(self.conv1_1(x)), inplace=False)
        x = F.relu(self.bn1_2(self.conv1_2(x)), inplace=False)
        x = self.pool1(x)
        x = self.drop1(x)

        # Block 2
        x = F.relu(self.bn2_1(self.conv2_1(x)), inplace=False)
        x = F.relu(self.bn2_2(self.conv2_2(x)), inplace=False)
        x = self.pool2(x)
        x = self.drop2(x)

        # Block 3
        x = F.relu(self.bn3_1(self.conv3_1(x)), inplace=False)
        x = F.relu(self.bn3_2(self.conv3_2(x)), inplace=False)
        x = F.relu(self.bn3_3(self.conv3_3(x)), inplace=False)
        x = self.pool3(x)
        x = self.drop3(x)

        return x

    def forward(self, x):
        x = self._forward_conv_blocks(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x), inplace=False)
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x


class WideResidualBlock(nn.Module):
    """Residual block with optional downsampling for Wide-ResNet style architecture."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=False)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)  # Residual connection
        out = F.relu(out, inplace=False)
        return out


class WideResNet(nn.Module):
    """
    Wide ResNet for CIFAR-10 - excellent performance with residual connections.

    Uses wider layers (more channels) and residual connections for better
    gradient flow and higher accuracy.
    """
    def __init__(self, n_in, n_hidden, n_out, width_factor=2):
        super().__init__()
        _, C, H, W = n_in

        # Initial convolution
        self.conv1 = nn.Conv2d(C, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # Residual blocks with increasing channels
        self.layer1 = self._make_layer(16, 16 * width_factor, 2, stride=1)
        self.layer2 = self._make_layer(16 * width_factor, 32 * width_factor, 2, stride=2)
        self.layer3 = self._make_layer(32 * width_factor, 64 * width_factor, 2, stride=2)

        # Global average pooling and classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64 * width_factor, n_out)
        self.dropout = nn.Dropout(0.3)

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(WideResidualBlock(in_channels, out_channels, stride))
        for _ in range(num_blocks - 1):
            layers.append(WideResidualBlock(out_channels, out_channels, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=False)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


# Convenience function to get recommended model
def get_improved_cnn(architecture='improved', n_in=(1, 3, 32, 32), n_hidden=256, n_out=10):
    """
    Get an improved CNN architecture.

    Args:
        architecture: 'improved', 'vgg', or 'wide_resnet'
        n_in: Input shape (batch_size, channels, height, width)
        n_hidden: Hidden dimension (used for some architectures)
        n_out: Number of output classes

    Returns:
        PyTorch model

    Recommended:
    - 'improved': Good balance of speed and accuracy (~85-88% on CIFAR-10)
    - 'vgg': Higher capacity, better accuracy (~88-90% on CIFAR-10)
    - 'wide_resnet': Best accuracy but slower (~90-92% on CIFAR-10)
    """
    if architecture == 'improved':
        return ImprovedCNN(n_in, n_hidden, n_out)
    elif architecture == 'vgg':
        return VGGStyleCNN(n_in, n_hidden, n_out)
    elif architecture == 'wide_resnet':
        return WideResNet(n_in, n_hidden, n_out)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")


if __name__ == "__main__":
    # Test the models
    batch_size = 4
    x = torch.randn(batch_size, 3, 32, 32)

    print("Testing ImprovedCNN...")
    model1 = ImprovedCNN((batch_size, 3, 32, 32), 256, 10)
    out1 = model1(x)
    print(f"  Output shape: {out1.shape}")
    print(f"  Parameters: {sum(p.numel() for p in model1.parameters()):,}")

    print("\nTesting VGGStyleCNN...")
    model2 = VGGStyleCNN((batch_size, 3, 32, 32), 256, 10)
    out2 = model2(x)
    print(f"  Output shape: {out2.shape}")
    print(f"  Parameters: {sum(p.numel() for p in model2.parameters()):,}")

    print("\nTesting WideResNet...")
    model3 = WideResNet((batch_size, 3, 32, 32), 256, 10)
    out3 = model3(x)
    print(f"  Output shape: {out3.shape}")
    print(f"  Parameters: {sum(p.numel() for p in model3.parameters()):,}")


In [ ]:
# This is the COMPLETE classifier.py with ImprovedCNN
# Copy this entire cell to replace your %%writefile classifier.py cell

%%writefile classifier.py
import sys
sys.dont_write_bytecode = True

import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as tv_models
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader

# Auto-install opacus if not present (for DP-SGD support)
try:
    import opacus  # noqa: F401
except ImportError:
    print("opacus not found. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opacus"])
    print("opacus installed successfully!")
    import opacus  # noqa: F401


# ---------------------------
#   Minibatch iterator
# ---------------------------
def iterate_minibatches(inputs, targets, batch_size, shuffle=True):
    assert len(inputs) == len(targets)
    if shuffle:
        indices = np.arange(len(inputs))
        np.random.shuffle(indices)

    start_idx = None
    for start_idx in range(0, len(inputs) - batch_size + 1, batch_size):
        if shuffle:
            excerpt = indices[start_idx:start_idx + batch_size]
        else:
            excerpt = slice(start_idx, start_idx + batch_size)
        yield inputs[excerpt], targets[excerpt]

    if start_idx is not None and start_idx + batch_size < len(inputs):
        excerpt = indices[start_idx + batch_size:] if shuffle else slice(start_idx + batch_size, len(inputs))
        yield inputs[excerpt], targets[excerpt]


# ---------------------------
#  CIFAR-10 Data Augmentation
# ---------------------------
def get_cifar10_transforms(train=True):
    """
    Strong CIFAR-10 augmentation.
    NOTE: In this repo, use_augmentation should be False for fair MG vs DP comparison,
    and DP path explicitly disables augmentation (train_transform=None).
    """
    if train:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                               (0.2470, 0.2435, 0.2616)),
        ])
    else:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                               (0.2470, 0.2435, 0.2616)),
        ])


# ---------------------------
#           Models
# ---------------------------
class CNNNet(nn.Module):
    """
    IMPROVED CNN with batch normalization, dropout, and deeper architecture.

    Expected accuracy on CIFAR-10: ~85-88% (vs ~70-75% for original)

    Architecture:
    Conv(32)x2 -> BN -> ReLU -> Pool -> Dropout(0.2)
    Conv(64)x2 -> BN -> ReLU -> Pool -> Dropout(0.3)
    Conv(128) -> BN -> ReLU -> Pool -> Dropout(0.4)
    FC(256) -> ReLU -> Dropout(0.5) -> FC(10)

    IMPORTANT for Opacus: no in-place activations.
    """
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        _, C, H, W = n_in

        # First convolutional block: 32 filters
        self.conv1a = nn.Conv2d(C, 32, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(32)
        self.conv1b = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.drop1 = nn.Dropout(0.2)

        # Second convolutional block: 64 filters
        self.conv2a = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(64)
        self.conv2b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.drop2 = nn.Dropout(0.3)

        # Third convolutional block: 128 filters
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.drop3 = nn.Dropout(0.4)

        # Calculate flattened dimension after convolutions
        with torch.no_grad():
            dummy = torch.zeros(1, C, H, W)
            x = self.pool1(F.relu(self.bn1b(self.conv1b(F.relu(self.bn1a(self.conv1a(dummy)))))))
            x = self.pool2(F.relu(self.bn2b(self.conv2b(F.relu(self.bn2a(self.conv2a(x)))))))
            x = self.pool3(F.relu(self.bn3(self.conv3(x))))
            flat_dim = x.numel()

        # Fully connected layers
        self.fc1 = nn.Linear(flat_dim, 256)
        self.drop_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, n_out)

    def forward(self, x):
        # First block
        x = F.relu(self.bn1a(self.conv1a(x)), inplace=False)
        x = F.relu(self.bn1b(self.conv1b(x)), inplace=False)
        x = self.pool1(x)
        x = self.drop1(x)

        # Second block
        x = F.relu(self.bn2a(self.conv2a(x)), inplace=False)
        x = F.relu(self.bn2b(self.conv2b(x)), inplace=False)
        x = self.pool2(x)
        x = self.drop2(x)

        # Third block
        x = F.relu(self.bn3(self.conv3(x)), inplace=False)
        x = self.pool3(x)
        x = self.drop3(x)

        # Classifier
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x), inplace=False)
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x


class CIFARResNet18(nn.Module):
    """ResNet-18 adapted for CIFAR-10 (32x32)."""
    def __init__(self, n_in, n_out):
        super().__init__()
        base = tv_models.resnet18(weights=None)
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        base.fc = nn.Linear(base.fc.in_features, n_out)
        self.net = base

    def forward(self, x):
        return self.net(x)


class CIFARResNet34(nn.Module):
    """ResNet-34 adapted for CIFAR-10 (32x32)."""
    def __init__(self, n_in, n_out):
        super().__init__()
        base = tv_models.resnet34(weights=None)
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        base.fc = nn.Linear(base.fc.in_features, n_out)
        self.net = base

    def forward(self, x):
        return self.net(x)


class MLPNet(nn.Module):
    """Deeper MLP: 3072 -> 1024 -> 512 -> 256 -> 10 (no in-place ops)."""
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        D = n_in[1] if len(n_in) > 1 else n_in[0]
        self.layers = nn.Sequential(
            nn.Linear(D, 1024),
            nn.ReLU(inplace=False),
            nn.Dropout(0.2),

            nn.Linear(1024, 512),
            nn.ReLU(inplace=False),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(inplace=False),
            nn.Dropout(0.2),

            nn.Linear(256, n_out),
        )

    def forward(self, x):
        if x.dim() > 2:
            x = x.reshape(x.size(0), -1)
        return self.layers(x)


class SoftmaxNet(nn.Module):
    """Single linear layer: Input -> logits."""
    def __init__(self, n_in, n_out):
        super().__init__()
        D = n_in[1] if len(n_in) > 1 else n_in[0]
        self.out = nn.Linear(D, n_out)

    def forward(self, x):
        if x.dim() > 2:
            x = x.reshape(x.size(0), -1)
        return self.out(x)


# ---------------------------
#    Online Augmentation
# ---------------------------
def apply_augmentation_batch(batch_np, transform, is_cnn):
    """
    Apply torchvision transforms to a numpy batch.
    NOTE: In this repo, preprocessing already normalizes per-pixel using train mean/std.
    For fair MG vs DP runs, set use_augmentation=False in your experiments.
    """
    if (not is_cnn) or (transform is None):
        return torch.from_numpy(batch_np).float()

    augmented = []
    for img in batch_np:
        img_hwc = np.transpose(img, (1, 2, 0))

        # If appears normalized (negatives), attempt to denorm using CIFAR-10 channel stats.
        # This is not a perfect inverse of mlLeaks per-pixel normalization,
        # so for clean experiments keep use_augmentation=False.
        if img_hwc.min() < 0:
            mean = np.array([0.4914, 0.4822, 0.4465])
            std = np.array([0.2470, 0.2435, 0.2616])
            img_hwc = img_hwc * std + mean
            img_hwc = np.clip(img_hwc * 255, 0, 255).astype(np.uint8)
        else:
            img_hwc = np.clip(img_hwc, 0, 255).astype(np.uint8)

        img_t = transform(img_hwc)
        augmented.append(img_t)

    return torch.stack(augmented, dim=0)


# ---------------------------
#          Evaluation
# ---------------------------
@torch.no_grad()
def _predict_batches(model, inputs, targets, batch_size, device, is_cnn):
    model.eval()
    preds = []
    if batch_size > len(targets):
        batch_size = len(targets)

    for xb_np, _ in iterate_minibatches(inputs, targets, batch_size, shuffle=False):
        xb = torch.from_numpy(xb_np).to(device=device, dtype=torch.float32)
        logits = model(xb)
        pred = torch.argmax(logits, dim=1)
        preds.append(pred.detach().cpu().numpy())

    return np.concatenate(preds, axis=0) if len(preds) > 0 else np.array([], dtype=np.int64)


@torch.no_grad()
def eval_accuracy(model, inputs, targets, batch_size, device, is_cnn) -> float:
    if inputs is None or targets is None or len(targets) == 0:
        return float("nan")
    preds = _predict_batches(model, inputs, targets, batch_size, device, is_cnn)
    return float(accuracy_score(targets, preds))


# ---------------------------
#          Training
# ---------------------------
def train_model(
    dataset,
    n_hidden=50,
    batch_size=100,
    epochs=100,
    learning_rate=0.01,
    model="cnn",
    l2_ratio=1e-7,
    *,
    mg_kappa: float = 0.0,
    mg_mode: str = "gauss",
    target_test_acc=None,
    eval_every: int = 1,
    use_augmentation: bool = True,
    trainer: str = "standard",        # "standard", "mg", or "dp"
    dp_noise_multiplier: float = 1.0, # DP noise multiplier σ
    dp_max_grad_norm: float = 1.0,    # clipping norm C
    dp_target_delta: float = 1e-5     # δ
):
    """
    Train a model with optional MG noise (trainer="mg") or DP-SGD (trainer="dp").

    DP-SGD path:
    - uses Opacus PrivacyEngine
    - applies ModuleValidator.fix(net) for Opacus compatibility
    - forces all nn.ReLU to inplace=False
    - uses a PyTorch DataLoader as required by Opacus
    """
    train_x, train_y, test_x, test_y = dataset

    is_cnn = model in {"cnn", "cnn2", "Droppcnn", "Droppcnn2", "resnet18", "resnet34"}

    # Flatten for non-CNN models
    if not is_cnn:
        print("Flattening inputs for MLP/Softmax...")
        if train_x.ndim > 2:
            train_x = train_x.reshape(train_x.shape[0], -1)
        if test_x is not None and getattr(test_x, "ndim", 0) > 2:
            test_x = test_x.reshape(test_x.shape[0], -1)

    n_in = train_x.shape
    n_out = len(np.unique(train_y))

    if batch_size > len(train_y):
        batch_size = len(train_y)

    print(f"Building model with {len(train_x)} training data, {n_out} classes...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model
    if is_cnn:
        if model == "resnet18":
            print("Using ResNet-18...")
            net = CIFARResNet18(n_in=n_in, n_out=n_out)
        elif model == "resnet34":
            print("Using ResNet-34...")
            net = CIFARResNet34(n_in=n_in, n_out=n_out)
        else:
            print("Using IMPROVED CNN...")
            net = CNNNet(n_in=n_in, n_hidden=n_hidden, n_out=n_out)
    elif model == "nn":
        print("Using deeper MLP (3 hidden layers: 1024->512->256)...")
        net = MLPNet(n_in=n_in, n_hidden=n_hidden, n_out=n_out)
    else:
        print("Using softmax regression...")
        net = SoftmaxNet(n_in=n_in, n_out=n_out)

    net.to(device)

    # Augmentation transforms (disabled for DP; recommended off for fair MG vs DP here)
    train_transform = get_cifar10_transforms(train=True) if (is_cnn and use_augmentation and trainer != "dp") else None

    # Loss & optimizer
    criterion = nn.CrossEntropyLoss()

    if model in {"resnet18", "resnet34"}:
        optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=l2_ratio)
        scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 120, 160], gamma=0.2)
    else:
        optimizer = optim.Adam(net.parameters(), lr=learning_rate, weight_decay=l2_ratio)
        scheduler = None

    # ---------------- DP-SGD Setup ----------------
    privacy_engine = None
    train_loader_dp = None

    if trainer == "dp":
        try:
            from opacus import PrivacyEngine
            from opacus.validators import ModuleValidator

            print(f"Enabling DP-SGD: σ={dp_noise_multiplier}, C={dp_max_grad_norm}, δ={dp_target_delta}")

            # Build DataLoader for Opacus
            train_dataset = TensorDataset(
                torch.from_numpy(train_x).float(),
                torch.from_numpy(train_y).long()
            )
            train_loader_pt = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
                drop_last=True,  # helps Opacus keep consistent sample rate
                generator=torch.Generator().manual_seed(42)
            )

            # Validate + auto-fix for Opacus
            errs = ModuleValidator.validate(net, strict=False)
            if len(errs) > 0:
                print("ModuleValidator found issues; applying automatic fixes...")
                net = ModuleValidator.fix(net)
                net.to(device)

                # CRITICAL FIX: Recreate optimizer after model is fixed
                # The old optimizer has references to old parameters
                if model in {"resnet18", "resnet34"}:
                    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=l2_ratio)
                    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 120, 160], gamma=0.2)
                else:
                    optimizer = optim.Adam(net.parameters(), lr=learning_rate, weight_decay=l2_ratio)
                    scheduler = None

            # Force ALL ReLUs to non-inplace (extra safety)
            for m in net.modules():
                if isinstance(m, nn.ReLU):
                    m.inplace = False

            privacy_engine = PrivacyEngine()

            # Opacus API differs across versions; try common signatures
            try:
                net, optimizer, train_loader_dp = privacy_engine.make_private(
                    module=net,
                    optimizer=optimizer,
                    data_loader=train_loader_pt,
                    noise_multiplier=dp_noise_multiplier,
                    max_grad_norm=dp_max_grad_norm,
                )
            except TypeError:
                # Older/newer variants sometimes require poisson_sampling explicitly
                net, optimizer, train_loader_dp = privacy_engine.make_private(
                    module=net,
                    optimizer=optimizer,
                    data_loader=train_loader_pt,
                    noise_multiplier=dp_noise_multiplier,
                    max_grad_norm=dp_max_grad_norm,
                    poisson_sampling=False,
                )

            print("DP-SGD initialized successfully.")

        except Exception as e:
            print(f"ERROR: Failed to initialize DP-SGD: {e}")
            print("Falling back to standard training.")
            trainer = "standard"
            privacy_engine = None
            train_loader_dp = None

    # ---------------- Training loop ----------------
    if trainer == "dp":
        mode_str = f"DP-SGD (σ={dp_noise_multiplier})"
    elif trainer == "mg" or mg_kappa > 0:
        mode_str = f"MG (κ={mg_kappa})"
    else:
        mode_str = "Standard"

    print(f"Training for {epochs} epochs | Mode: {mode_str} | Augmentation: {bool(train_transform is not None)}")

    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        if trainer == "dp" and train_loader_dp is not None:
            # DP-SGD training (no augmentation, no MG)
            for xb, yb in train_loader_dp:
                xb = xb.to(device=device, dtype=torch.float32)
                yb = yb.to(device=device, dtype=torch.long)
                optimizer.zero_grad(set_to_none=True)
                logits = net(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
                running_loss += float(loss.item())
        else:
            # Standard or MG training
            for xb_np, yb_np in iterate_minibatches(train_x, train_y, batch_size, shuffle=True):
                xb = apply_augmentation_batch(xb_np, train_transform, is_cnn).to(device=device, dtype=torch.float32)
                yb = torch.from_numpy(yb_np).to(device=device, dtype=torch.long)

                # MG noise (only in mg mode)
                if (trainer == "mg" or mg_kappa > 0) and mg_kappa > 0.0:
                    if mg_mode == "lognorm":
                        mask = torch.exp(torch.randn_like(xb) * mg_kappa - 0.5 * (mg_kappa ** 2))
                    else:
                        mask = 1.0 + mg_kappa * torch.randn_like(xb)
                    xb = xb * mask

                optimizer.zero_grad(set_to_none=True)
                logits = net(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
                running_loss += float(loss.item())

        if scheduler is not None:
            scheduler.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch+1}, train loss {round(running_loss, 3)}")

        # Early stopping by test accuracy target
        if target_test_acc is not None and (epoch % max(1, eval_every) == 0):
            acc = eval_accuracy(net, test_x, test_y, batch_size, device, is_cnn)
            print(f"[eval] epoch {epoch+1} test acc = {acc*100:.2f}%")
            if acc >= target_test_acc:
                print(f"Reached target test accuracy {target_test_acc*100:.1f}%. Stopping.")
                break

    # DP: compute epsilon if available
    if trainer == "dp" and privacy_engine is not None:
        try:
            epsilon = privacy_engine.get_epsilon(delta=dp_target_delta)
            print(f"DP-SGD training complete. ε = {epsilon:.2f} (δ = {dp_target_delta})")
        except Exception as e:
            print(f"Warning: Could not compute epsilon: {e}")

    # Final test report
    if test_x is not None and test_y is not None and len(test_y) > 0:
        print("Testing...")
        pred_y = _predict_batches(net, test_x, test_y, batch_size, device, is_cnn=is_cnn)
        if len(pred_y) > 0:
            acc = accuracy_score(test_y, pred_y)
            print(f"Testing Accuracy: {acc:.4f}")
            print("More detailed results:")
            print(classification_report(test_y, pred_y))
        else:
            print("No test batches produced.")

    return net


# ---------------------------
#          Predict
# ---------------------------
def predict(model_file, test_x, test_y, batch_size=100):
    """Load a model and predict on test data."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # You'll need to know the model architecture to load properly
    # This is a simplified version - adjust as needed
    print(f"Loading model from {model_file}")
    # Implementation depends on how you save your models
    pass


In [ ]:
# deeplearning.py - Updated to support augmentation parameter
%%writefile deeplearning.py
import sys
sys.dont_write_bytecode = True

import numpy as np
import torch

from classifier import train_model, iterate_minibatches

np.random.seed(21312)
torch.manual_seed(21312)


def _probs_from_batches(model, inputs, batch_size, device):
    """
    Run model on inputs in batches (no shuffle), return (N, num_classes) softmax probs.
    """
    model.eval()
    probs_parts = []
    if batch_size > len(inputs):
        batch_size = len(inputs)
    with torch.no_grad():
        for xb_np, _ in iterate_minibatches(inputs, np.zeros(len(inputs), dtype=np.int32),
                                           batch_size, shuffle=False):
            xb = torch.from_numpy(xb_np).to(device=device, dtype=torch.float32)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            probs_parts.append(probs.cpu().numpy())
    if len(probs_parts) == 0:
        return np.zeros((0, 0), dtype=np.float32)
    return np.vstack(probs_parts).astype('float32')


def train_target_model(dataset,
                       epochs=100,
                       batch_size=100,
                       learning_rate=0.01,
                       l2_ratio=1e-7,
                       n_hidden=50,
                       model='nn',
                       mg_kappa=0.0,
                       mg_mode='gauss',
                       target_test_acc=None,
                       eval_every=1,
                       use_augmentation=True,
                       trainer='standard',           # NEW
                       dp_noise_multiplier=1.0,      # NEW
                       dp_max_grad_norm=1.0,         # NEW
                       dp_target_delta=1e-5):        # NEW
    """
    Train a target model and build an attack dataset.

    Args:
      dataset: (train_x, train_y, test_x, test_y)
      trainer: "standard", "mg", or "dp"
      use_augmentation: if True, apply strong CIFAR-10 augmentation during training
      mg_kappa: MG noise strength (when trainer="mg")
      dp_noise_multiplier, dp_max_grad_norm, dp_target_delta: DP-SGD params (when trainer="dp")

    Returns:
      attack_x: (N_total, num_classes) softmax probs
      attack_y: (N_total,) 1=member, 0=non-member
      trained_model: nn.Module
    """
    train_x, train_y, test_x, test_y = dataset

    # Train the model (pass through all parameters)
    trained_model = train_model(
        (train_x, train_y, test_x, test_y),
        n_hidden=n_hidden,
        batch_size=batch_size,
        epochs=epochs,
        learning_rate=learning_rate,
        model=model,
        l2_ratio=l2_ratio,
        mg_kappa=mg_kappa,
        mg_mode=mg_mode,
        target_test_acc=target_test_acc,
        eval_every=eval_every,
        use_augmentation=use_augmentation,
        trainer=trainer,
        dp_noise_multiplier=dp_noise_multiplier,
        dp_max_grad_norm=dp_max_grad_norm,
        dp_target_delta=dp_target_delta
    )

    try:
        device = next(trained_model.parameters()).device
    except StopIteration:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Build attack dataset
    attack_x_parts, attack_y_parts = [], []

    # Members (label 1)
    members_probs = _probs_from_batches(trained_model, train_x, batch_size, device)
    if members_probs.size > 0:
        attack_x_parts.append(members_probs)
        attack_y_parts.append(np.ones(len(members_probs), dtype=np.int32))

    # Non-members (label 0)
    nonmembers_probs = _probs_from_batches(trained_model, test_x, batch_size, device)
    if nonmembers_probs.size > 0:
        attack_x_parts.append(nonmembers_probs)
        attack_y_parts.append(np.zeros(len(nonmembers_probs), dtype=np.int32))

    if len(attack_x_parts) == 0:
        attack_x = np.zeros((0, 0), dtype=np.float32)
        attack_y = np.zeros((0,), dtype=np.int32)
    else:
        attack_x = np.vstack(attack_x_parts).astype('float32')
        attack_y = np.concatenate(attack_y_parts).astype('int32')

    return attack_x, attack_y, trained_model


In [ ]:
# mlLeaks.py - Updated with better MLP capacity and augmentation
%%writefile mlLeaks.py
import sys
sys.dont_write_bytecode = True

import os
import pickle
import argparse
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.utils import shuffle as sk_shuffle
from sklearn.metrics import precision_score, recall_score

import deeplearning as dp
import classifier

# -------------------------
# CLI
# -------------------------
parser = argparse.ArgumentParser()
parser.add_argument('--adv', default='1', choices=['1','2','3'], help='Which adversary 1,2,3')
parser.add_argument('--dataset', default='CIFAR10', choices=['CIFAR10','News'], help='Which dataset for target/shadow')
parser.add_argument('--classifierType', default='cnn', choices=['cnn','nn','softmax','resnet18', 'resnet34'], help='Classifier type for dataset1')
parser.add_argument('--dataset2', default='News', choices=['CIFAR10','News'], help='Second dataset for adversary2')
parser.add_argument('--classifierType2', default='nn', choices=['cnn','nn','softmax','resnet18', 'resnet34'], help='Classifier type for dataset2')
parser.add_argument('--dataFolderPath', default='../data', help='Base path to save preprocessed data')
parser.add_argument('--pathToLoadData', default='../data/datasets/cifar-10-batches-py', help='Path to CIFAR pickles')
parser.add_argument('--num_epoch', type=int, default=50, help='Epochs for training shadow/target models')
parser.add_argument('--preprocessData', action='store_true', help='Run preprocessing (otherwise load saved .npz)')
parser.add_argument('--trainTargetModel', action='store_true', help='Train target model (otherwise load saved)')
parser.add_argument('--trainShadowModel', action='store_true', help='Train shadow model (otherwise load saved)')
parser.add_argument('--top_k', type=int, default=3, help='Top-k probabilities to keep for attack features')
parser.add_argument('--attack_clf_sklearn', action='store_true', help='Train attack classifier with sklearn LogisticRegression (default True)')
parser.add_argument('--random_seed', type=int, default=42, help='Random seed')
#### MG #####
parser.add_argument('--trainer', default='standard', choices=['standard','mg','dp'],
                    help='Training mode for target/shadow models')
parser.add_argument('--mg_kappa', type=float, default=0.0,
                    help='Multiplicative Gaussian noise strength (0 disables)')
parser.add_argument('--mg_mode', default='gauss', choices=['gauss','lognorm'],
                    help="MG noise: 'gauss' => (1 + kappa*Z), 'lognorm' => exp(kappa*Z - 0.5*kappa^2)")
parser.add_argument('--target_test_acc', type=float, default=None,
                    help='Early-stop when test accuracy >= this value (e.g., 0.35)')
parser.add_argument('--eval_every', type=int, default=1,
                    help='Evaluate test accuracy every N epochs for early-stop')
#### Augmentation ####
parser.add_argument('--use_augmentation', action='store_true', default=True,
                    help='Enable data augmentation for CIFAR-10 (default: True)')
parser.add_argument('--no_augmentation', dest='use_augmentation', action='store_false',
                    help='Disable data augmentation')
#### DP #####
parser.add_argument('--dp_noise_multiplier', type=float, default=1.0,
                    help='DP-SGD noise multiplier (sigma).')
parser.add_argument('--dp_max_grad_norm', type=float, default=1.0,
                    help='Per-sample gradient clip norm C.')
parser.add_argument('--dp_target_epsilon', type=float, default=None,
                    help='Stop early when epsilon <= this (optional).')
parser.add_argument('--dp_target_delta', type=float, default=1e-5,
                    help='Delta for (epsilon, delta)-DP accounting.')
parser.add_argument('--dp_max_epochs', type=int, default=None,
                    help='Optional hard cap on DP epochs (overrides --num_epoch if set).')
parser.add_argument('--shadow_trainer', default=None, choices=[None,'standard','mg','dp'],
                    help='If set, overrides --trainer for the shadow model.')

opt, _ = parser.parse_known_args()

np.random.seed(opt.random_seed)
random.seed(opt.random_seed)
torch.manual_seed(opt.random_seed)


def clip_top_k(data, top=3):
    """Keep only the top-k values of each row (descending). Returns (N,top)."""
    if top is None:
        return data
    res = [sorted(row, reverse=True)[:top] for row in data]
    return np.array(res, dtype=np.float32)


def read_cifar10(data_path):
    """Load original CIFAR-10 Python pickles. Returns X (N,3072), y (N,)."""
    X_parts = []
    y_parts = []
    for i in range(1, 6):
        p = Path(data_path) / f"data_batch_{i}"
        with open(p, 'rb') as f:
            batch = pickle.load(f, encoding='latin1')
        X_parts.append(np.array(batch['data']))
        y_parts.append(np.array(batch['labels']))
    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    with open(Path(data_path) / 'test_batch', 'rb') as f:
        test_batch = pickle.load(f, encoding='latin1')
    Xtest = np.array(test_batch['data'])
    ytest = np.array(test_batch['labels'])
    return X, y, Xtest, ytest


def reshape_and_normalize_cifar(train_flat, test_flat):
    """From (N,3072) -> (N,3,32,32) channels-first, normalized by train mean/std."""
    def reshape(raw):
        raw = np.dstack((raw[:, :1024], raw[:, 1024:2048], raw[:, 2048:]))
        raw = raw.reshape((raw.shape[0], 32, 32, 3)).transpose(0,3,1,2)
        return raw.astype(np.float32)
    train_img = reshape(train_flat)
    test_img = reshape(test_flat)
    mean = np.mean(train_img, axis=0)
    std = np.std(train_img, axis=0).clip(min=1.0)
    train_scaled = (train_img - mean) / std
    test_scaled = (test_img - mean) / std
    return train_scaled.astype(np.float32), test_scaled.astype(np.float32)


def preprocess_news(all_texts_train, all_texts_test, max_features=None):
    """TF-IDF vectorize and normalize."""
    vectorizer = TfidfVectorizer(max_features=max_features)
    combined = np.concatenate([all_texts_train, all_texts_test], axis=0)
    X = vectorizer.fit_transform(combined).toarray()
    n_train = len(all_texts_train)
    train = X[:n_train]
    test = X[n_train:]
    mean = np.mean(train, axis=0)
    std = np.std(train, axis=0).clip(min=1.0)
    return ((train - mean) / std).astype(np.float32), ((test - mean) / std).astype(np.float32)


def list_shuffle_split(X, y, cluster):
    """Shuffle and split into 4 groups of size cluster each."""
    arr = list(zip(X, y))
    random.shuffle(arr)
    Xs, ys = zip(*arr)
    Xs = np.array(Xs)
    ys = np.array(ys)
    total_needed = cluster * 4
    if len(Xs) < total_needed:
        raise ValueError(f"Not enough data for cluster={cluster}; need {total_needed}, got {len(Xs)}")
    a, b = Xs, ys
    return (a[:cluster], b[:cluster],
            a[cluster:2*cluster], b[cluster:2*cluster],
            a[2*cluster:3*cluster], b[2*cluster:3*cluster],
            a[3*cluster:4*cluster], b[3*cluster:4*cluster])


def save_npz(path, *arrays):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, *arrays)


def load_npz(path):
    with np.load(path) as f:
        return [f[f"arr_{i}"] for i in range(len(f.files))]


def initialize_data(dataset, origin_path, data_folder='./data/'):
    data_folder = Path(data_folder)
    path_out = data_folder / dataset / 'Preprocessed'
    path_out.mkdir(parents=True, exist_ok=True)

    if dataset == 'CIFAR10':
        print("Loading CIFAR-10 from", origin_path)
        X, y, Xtest, ytest = read_cifar10(origin_path)
        cluster = 10520
        X_all = np.concatenate([X, Xtest], axis=0)
        y_all = np.concatenate([y, ytest], axis=0)
        toTrain, toTrainLabel, shadow, shadowLabel, toTest, toTestLabel, shadowTest, shadowTestLabel = list_shuffle_split(X_all, y_all, cluster)
        toTrainSave, toTestSave = reshape_and_normalize_cifar(toTrain, toTest)
        shadowSave, shadowTestSave = reshape_and_normalize_cifar(shadow, shadowTest)
    else:  # News
        from sklearn.datasets import fetch_20newsgroups
        print("Fetching 20 newsgroups")
        newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
        newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers','footers','quotes'))
        texts_train = newsgroups_train.data
        texts_test = newsgroups_test.data
        labels_train = newsgroups_train.target
        labels_test = newsgroups_test.target
        X_all_texts = np.concatenate([texts_train, texts_test], axis=0)
        y_all = np.concatenate([labels_train, labels_test], axis=0)
        cluster = 4500
        toTrain, toTrainLabel, shadow, shadowLabel, toTest, toTestLabel, shadowTest, shadowTestLabel = list_shuffle_split(X_all_texts, y_all, cluster)
        toTrainSave, toTestSave = preprocess_news(toTrain, toTest, max_features=None)
        shadowSave, shadowTestSave = preprocess_news(shadow, shadowTest, max_features=None)

    save_npz(str(path_out / 'targetTrain.npz'), toTrainSave, toTrainLabel)
    save_npz(str(path_out / 'targetTest.npz'), toTestSave, toTestLabel)
    save_npz(str(path_out / 'shadowTrain.npz'), shadowSave, shadowLabel)
    save_npz(str(path_out / 'shadowTest.npz'), shadowTestSave, shadowTestLabel)
    print("Saved preprocessed data to:", path_out)


# -------------------------
# Train + save target/shadow models
# -------------------------
def initialize_target_model(dataset, num_epoch, data_folder='./data/', model_folder='./model/',
                           classifier_type='cnn', batch_size=100, learning_rate=1e-3, top_k=3):
    data_path = Path(data_folder) / dataset / 'Preprocessed'
    attacker_out = Path(data_folder) / dataset / 'attackerModelData'
    model_out = Path(model_folder) / dataset
    attacker_out.mkdir(parents=True, exist_ok=True)
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"Training target model for {num_epoch} epochs...")
    target_train, target_train_label = load_npz(str(data_path / 'targetTrain.npz'))
    target_test, target_test_label = load_npz(str(data_path / 'targetTest.npz'))

    # CHANGED: n_hidden=1024 (was 100), use_augmentation enabled
    attack_x, attack_y, model = dp.train_target_model(
        dataset=(target_train.astype(np.float32), target_train_label.astype(np.int32),
                 target_test.astype(np.float32),  target_test_label.astype(np.int32)),
        epochs=num_epoch,
        batch_size=batch_size,
        learning_rate=learning_rate,
        l2_ratio=1e-7,
        n_hidden=1024,  # <-- INCREASED from 100
        model=classifier_type,
        mg_kappa=opt.mg_kappa if opt.trainer == 'mg' else 0.0,
        mg_mode=opt.mg_mode,
        target_test_acc=opt.target_test_acc,
        eval_every=opt.eval_every,
        use_augmentation=opt.use_augmentation  # <-- PASS THROUGH
    )

    attack_x = attack_x.astype(np.float32)
    attack_y = attack_y.astype(np.int32)
    save_npz(str(attacker_out / 'targetModelData.npz'), attack_x, attack_y)
    try:
        torch.save(model.state_dict(), str(model_out / 'targetModel.pth'))
    except Exception as e:
        print("Warning: unable to save model state_dict:", e)

    return attack_x, attack_y, model


def initialize_shadow_model(dataset, num_epoch, data_folder='./data/', model_folder='./model/',
                           classifier_type='cnn', batch_size=100, learning_rate=1e-3, top_k=3):
    data_path = Path(data_folder) / dataset / 'Preprocessed'
    attacker_out = Path(data_folder) / dataset / 'attackerModelData'
    model_out = Path(model_folder) / dataset
    attacker_out.mkdir(parents=True, exist_ok=True)
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"Training shadow model for {num_epoch} epochs...")
    shadow_train, shadow_train_label = load_npz(str(data_path / 'shadowTrain.npz'))
    shadow_test, shadow_test_label = load_npz(str(data_path / 'shadowTest.npz'))

    # CHANGED: n_hidden=1024, use_augmentation enabled
    attack_x, attack_y, model = dp.train_target_model(
        dataset=(shadow_train.astype(np.float32), shadow_train_label.astype(np.int32),
                 shadow_test.astype(np.float32),  shadow_test_label.astype(np.int32)),
        epochs=num_epoch,
        batch_size=batch_size,
        learning_rate=learning_rate,
        l2_ratio=1e-7,
        n_hidden=1024,  # <-- INCREASED from 100
        model=classifier_type,
        mg_kappa=opt.mg_kappa if opt.trainer == 'mg' else 0.0,
        mg_mode=opt.mg_mode,
        target_test_acc=opt.target_test_acc,
        eval_every=opt.eval_every,
        use_augmentation=opt.use_augmentation  # <-- PASS THROUGH
    )

    attack_x = attack_x.astype(np.float32)
    attack_y = attack_y.astype(np.int32)
    save_npz(str(attacker_out / 'shadowModelData.npz'), attack_x, attack_y)
    try:
        torch.save(model.state_dict(), str(model_out / 'shadowModel.pth'))
    except Exception as e:
        print("Warning: unable to save model state_dict:", e)

    return attack_x, attack_y, model


# -------------------------
# Load precomputed attack data
# -------------------------
def load_attack_data(dataset, kind='target', data_folder='./data/'):
    data_path = Path(data_folder) / dataset / 'attackerModelData'
    arr = load_npz(str(data_path / f'{kind}ModelData.npz'))
    return arr[0].astype(np.float32), arr[1].astype(np.int32)


# -------------------------
# Attack classifier training + evaluation
# -------------------------
def train_attack_classifier_and_eval(train_X, train_y, test_X, test_y, balance=True):
    """Train sklearn LogisticRegression on train_X/train_y and evaluate on test_X/test_y."""
    if balance:
        pos = np.where(train_y == 1)[0]
        neg = np.where(train_y == 0)[0]
        m = min(len(pos), len(neg))
        if m == 0:
            raise ValueError("One of classes empty in attack training data.")
        sel = np.concatenate([np.random.choice(pos, m, replace=False),
                             np.random.choice(neg, m, replace=False)])
        train_X_bal, train_y_bal = train_X[sel], train_y[sel]
    else:
        train_X_bal, train_y_bal = train_X, train_y

    train_X_bal, train_y_bal = sk_shuffle(train_X_bal, train_y_bal, random_state=opt.random_seed)

    clf = LogisticRegression(max_iter=2000, solver='lbfgs')
    clf.fit(train_X_bal, train_y_bal)

    preds = clf.predict(test_X)
    probs = clf.predict_proba(test_X)[:, 1] if hasattr(clf, "predict_proba") else None

    acc = accuracy_score(test_y, preds)
    auc = roc_auc_score(test_y, probs) if probs is not None else None
    prec = precision_score(test_y, preds, average="binary", zero_division=0)
    rec = recall_score(test_y, preds, average="binary", zero_division=0)

    print("Attack classifier — acc: {:.4f}  AUC: {}  Prec: {:.4f}  Rec: {:.4f}".format(
        acc, auc, prec, rec))
    print(classification_report(test_y, preds))

    return clf, {"acc": acc, "auc": auc, "prec": prec, "rec": rec}


# ----------------------------------------------------------------
# Orchestration
# ----------------------------------------------------------------
def generate_attack_data(dataset, classifierType, dataFolderPath, pathToLoadData,
                        num_epoch, preprocessData, trainTargetModel, trainShadowModel, top_k=3):
    if preprocessData:
        initialize_data(dataset, pathToLoadData, data_folder=dataFolderPath)

    if trainTargetModel:
        tX, tY, tmodel = initialize_target_model(dataset, num_epoch,
                                                 data_folder=dataFolderPath,
                                                 classifier_type=classifierType)
    else:
        tX, tY = load_attack_data(dataset, 'target', data_folder=dataFolderPath)
        tmodel = None

    if trainShadowModel:
        sX, sY, smodel = initialize_shadow_model(dataset, num_epoch,
                                                 data_folder=dataFolderPath,
                                                 classifier_type=classifierType)
    else:
        sX, sY = load_attack_data(dataset, 'shadow', data_folder=dataFolderPath)
        smodel = None

    tX_clipped = clip_top_k(tX, top=top_k)
    sX_clipped = clip_top_k(sX, top=top_k)
    return tX_clipped, tY, sX_clipped, sY, tmodel, smodel


def attacker_one(dataset='CIFAR10', classifierType='cnn', dataFolderPath='../data',
                pathToLoadData='../data/datasets/cifar-10-batches-py',
                num_epoch=50, preprocessData=True, trainTargetModel=True,
                trainShadowModel=True, top_k=3):
    tX, tY, sX, sY, tmodel, smodel = generate_attack_data(
        dataset, classifierType, dataFolderPath, pathToLoadData,
        num_epoch, preprocessData, trainTargetModel, trainShadowModel, top_k=top_k)
    print("Training attack classifier (train on SHADOW, evaluate on TARGET).")
    clf, metrics = train_attack_classifier_and_eval(sX, sY, tX, tY, balance=True)
    return clf, metrics["acc"], metrics["auc"]


def attack_epochs_curve(
    epochs_list=(10,20,30,40,50,60,80,100),
    dataset="CIFAR10",
    classifierType="cnn",
    dataFolderPath="../data",
    batch_size=100,
    learning_rate=1e-3,
    n_hidden=1024,  # <-- CHANGED default from 100
    top_k=3,
    trainer="mg",
    mg_kappa=0.0,
    mg_mode="gauss",
    use_augmentation=True
):
    """
    Sweep over epochs_list, training target+shadow for each E, then attack.
    Returns: {"epochs": [...], "precision": [...], "recall": [...]}
    """
    data_path = Path(dataFolderPath) / dataset / 'Preprocessed'

    # Validation: ensure preprocessing was done
    if not (data_path / 'targetTrain.npz').exists():
        raise FileNotFoundError(
            f"Preprocessed data not found at {data_path}. "
            "Run with --preprocessData first or check --dataFolderPath."
        )

    target_train, target_train_label = load_npz(str(data_path / 'targetTrain.npz'))
    target_test, target_test_label = load_npz(str(data_path / 'targetTest.npz'))
    shadow_train, shadow_train_label = load_npz(str(data_path / 'shadowTrain.npz'))
    shadow_test, shadow_test_label = load_npz(str(data_path / 'shadowTest.npz'))

    xs, precisions, recalls = [], [], []

    for E in epochs_list:
        print(f"\n=== Epoch budget: {E} ===")

        # Train TARGET
        tX, tY, _tmodel = dp.train_target_model(
            dataset=(target_train.astype(np.float32), target_train_label.astype(np.int32),
                     target_test.astype(np.float32), target_test_label.astype(np.int32)),
            epochs=E, batch_size=batch_size, learning_rate=learning_rate,
            l2_ratio=1e-7, n_hidden=n_hidden, model=classifierType,
            mg_kappa=(mg_kappa if trainer == "mg" else 0.0),
            mg_mode=mg_mode, target_test_acc=None, eval_every=1,
            use_augmentation=use_augmentation
        )

        # Train SHADOW
        sX, sY, _smodel = dp.train_target_model(
            dataset=(shadow_train.astype(np.float32), shadow_train_label.astype(np.int32),
                     shadow_test.astype(np.float32), shadow_test_label.astype(np.int32)),
            epochs=E, batch_size=batch_size, learning_rate=learning_rate,
            l2_ratio=1e-7, n_hidden=n_hidden, model=classifierType,
            mg_kappa=(mg_kappa if trainer == "mg" else 0.0),
            mg_mode=mg_mode, target_test_acc=None, eval_every=1,
            use_augmentation=use_augmentation
        )

        sXc = clip_top_k(sX, top=top_k)
        tXc = clip_top_k(tX, top=top_k)

        _, metrics = train_attack_classifier_and_eval(sXc, sY, tXc, tY, balance=True)

        xs.append(E)
        precisions.append(metrics["prec"])
        recalls.append(metrics["rec"])

    return {"epochs": xs, "precision": precisions, "recall": recalls}


if __name__ == "__main__":
    if opt.adv == '1':
        attacker_one(dataset=opt.dataset, classifierType=opt.classifierType,
                    dataFolderPath=opt.dataFolderPath, pathToLoadData=opt.pathToLoadData,
                    num_epoch=opt.num_epoch, preprocessData=opt.preprocessData,
                    trainTargetModel=opt.trainTargetModel, trainShadowModel=opt.trainShadowModel,
                    top_k=opt.top_k)


In [ ]:
# ============================================================
# CELL 1: Mount Drive + folders
# ============================================================
# (removed for repo) from google.colab import drive
# (removed for repo) drive.mount(...)
import os
PLOTS_DIR = "../figures"
DATA_DIR  = "../data"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(" Drive mounted")
print(" PLOTS_DIR:", PLOTS_DIR)
print(" DATA_DIR :", DATA_DIR)


In [ ]:
# ============================================================
# CELL 2: (RUN ONCE) Preprocess CIFAR-10 into mlLeaks format
# ============================================================
import mlLeaks

# IMPORTANT: update this to where CIFAR-10 python batches live in your Drive
CIFAR_ORIGIN = "../data/datasets/cifar-10-batches-py"  # <-- CHANGE if needed

mlLeaks.initialize_data(
    dataset="CIFAR10",
    origin_path=CIFAR_ORIGIN,
    data_folder=DATA_DIR,
)

print(" Data preprocessing complete")


In [ ]:
# ============================================================
# CELL B: Imports + load preprocessed data
# ============================================================
import os, gc, json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

import mlLeaks
import deeplearning as dp
from mlLeaks import load_npz, clip_top_k, train_attack_classifier_and_eval

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PLOTS_DIR = "../figures"
DATA_DIR  = "../data"
os.makedirs(PLOTS_DIR, exist_ok=True)

dataset = "CIFAR10"
classifierType = "cnn"
data_path = Path(DATA_DIR) / dataset / "Preprocessed"

print("Loading:", data_path)
target_train, target_train_label = load_npz(str(data_path / "targetTrain.npz"))
target_test,  target_test_label  = load_npz(str(data_path / "targetTest.npz"))
shadow_train, shadow_train_label = load_npz(str(data_path / "shadowTrain.npz"))
shadow_test,  shadow_test_label  = load_npz(str(data_path / "shadowTest.npz"))

print(" Loaded preprocessed CIFAR-10")


In [ ]:
# ============================================================
# CELL C: DP-only sweep (run this first)
#   - augmentation OFF (fair + avoids repo aug bugs)
#   - start small to verify it doesn't crash
# ============================================================
import math

def _free(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

# ---- DP CONFIG ----
epochs = 10               # start small; increase to 50 after it works
BATCH_SIZE = 128          # 128/256 usually good on T4; drop if OOM
LR = 1e-3
n_hidden = 100            # ignored for CNN but required by signature

USE_AUGMENTATION = False  # IMPORTANT
DP_MAX_GRAD_NORM = 1.0

# Start with a tiny sweep (sanity check)
dp_noise_multipliers = [0.8, 1.0, 1.5]

out_base = f"{PLOTS_DIR}/dp_only_cnn"

def train_and_eval_dp(sigma, top_k=3):
    print(f"\n{'='*70}\nDP sweep: sigma={sigma}\n{'='*70}")

    kwargs = dict(
        dataset=(target_train, target_train_label, target_test, target_test_label),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        n_hidden=n_hidden,
        model=classifierType,
        trainer="dp",
        use_augmentation=USE_AUGMENTATION,
        dp_noise_multiplier=float(sigma),
        dp_max_grad_norm=float(DP_MAX_GRAD_NORM),
    )

    # Train TARGET
    print("Training TARGET...")
    tX, tY, tmodel = dp.train_target_model(**kwargs)

    # Evaluate TARGET
    tmodel.eval()
    device = next(tmodel.parameters()).device
    with torch.no_grad():
        preds = tmodel(torch.from_numpy(target_test).to(device).float()).argmax(1).cpu().numpy()
    acc = accuracy_score(target_test_label, preds)
    print(f"✓ Target accuracy: {acc*100:.2f}%")

    _free(tmodel)

    # Train SHADOW
    print("Training SHADOW...")
    kwargs["dataset"] = (shadow_train, shadow_train_label, shadow_test, shadow_test_label)
    sX, sY, smodel = dp.train_target_model(**kwargs)
    _free(smodel)

    # Attack
    print("Running attack...")
    sXc = clip_top_k(sX, top=top_k)
    tXc = clip_top_k(tX, top=top_k)
    _, metrics = train_attack_classifier_and_eval(sXc, sY, tXc, tY, balance=True)

    result = dict(
        defense="dp",
        sigma=float(sigma),
        acc=float(acc),
        prec=float(metrics["prec"]),
        rec=float(metrics["rec"]),
        auc=float(metrics.get("auc", np.nan)),
        top_k=int(top_k),
    )
    print(f"✓ Attack: prec={result['prec']:.3f}, rec={result['rec']:.3f}, auc={result['auc']:.3f}")

    _free(sX, sY, tX, tY, sXc, tXc)
    return result

dp_results = []
for sigma in dp_noise_multipliers:
    dp_results.append(train_and_eval_dp(sigma, top_k=3))
    _free()

# Save DP results
dp_json = f"{out_base}_results.json"
with open(dp_json, "w") as f:
    json.dump(dp_results, f, indent=2)
print(" Saved:", dp_json)

# Quick DP plot: accuracy vs attack AUC/precision
dp_acc  = np.array([r["acc"]*100 for r in dp_results])
dp_prec = np.array([r["prec"] for r in dp_results])
dp_rec  = np.array([r["rec"]  for r in dp_results])
dp_auc  = np.array([r["auc"]  for r in dp_results])

# Sort by accuracy for nicer curves
idx = np.argsort(dp_acc)
dp_acc, dp_prec, dp_rec, dp_auc = dp_acc[idx], dp_prec[idx], dp_rec[idx], dp_auc[idx]

plt.figure(figsize=(6.5,4.8))
plt.plot(dp_acc, dp_prec, "s--", linewidth=2.5, markersize=7, label="Attack Precision")
plt.plot(dp_acc, dp_rec,  "o-",  linewidth=2.5, markersize=7, label="Attack Recall")
plt.xlabel("Target Accuracy (%)")
plt.ylabel("Attack metric")
plt.title("DP-only sweep (CNN)")
plt.grid(True, alpha=0.3, linestyle="--")
plt.legend()
out_plot = f"{out_base}_tradeoff.png"
plt.savefig(out_plot, dpi=300, bbox_inches="tight")
plt.show()
print(" Saved:", out_plot)


In [ ]:
# ============================================================
# CELL 3: MG vs DP experiment (CNN) + iso-accuracy comparison
#   Key fixes:
#   - use_augmentation=False for BOTH MG and DP (fairness + avoids broken aug)
#   - sort by accuracy before np.interp (interp requires increasing xp)
#   - save outputs (npz/csv + plots)
# ============================================================

import os, gc, json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

import deeplearning as dp
from mlLeaks import load_npz, clip_top_k, train_attack_classifier_and_eval

# -------------------- CONFIG --------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

dataset = "CIFAR10"
classifierType = "cnn"          # cnn is faster in this codebase
dataFolderPath = DATA_DIR
epochs = 50
n_hidden = 100                  # required by signature; ignored for CNN

out_base = f"{PLOTS_DIR}/mg_vs_dp_cnn_isoacc"
DP_MAX_GRAD_NORM = 1.0

# Sweeps (tweak if you need overlap)
mg_kappas = [0.2, 0.6, 1.0, 1.4, 1.8, 2.5, 3.0]
#mg_kappas = [0.2, 0.6, 1.0, 1.4, 1.8, 2.5]
#mg_kappas = [0.2, 0.5, 0.8, 1.2, 1.5]
#dp_noise_multipliers = [0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0]
dp_noise_multipliers = [0.2, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0]


# Training hyperparams
BATCH_SIZE = 100
LR = 1e-3

# IMPORTANT: turn OFF augmentation for both (fair + avoids broken augmentation)
USE_AUGMENTATION = False

print("DEVICE:", "cuda" if torch.cuda.is_available() else "cpu")
print("USE_AUGMENTATION:", USE_AUGMENTATION)

# -------------------- LOAD DATA --------------------
data_path = Path(dataFolderPath) / dataset / "Preprocessed"
print("Loading preprocessed data from:", data_path)

target_train, target_train_label = load_npz(str(data_path / "targetTrain.npz"))
target_test,  target_test_label  = load_npz(str(data_path / "targetTest.npz"))
shadow_train, shadow_train_label = load_npz(str(data_path / "shadowTrain.npz"))
shadow_test,  shadow_test_label  = load_npz(str(data_path / "shadowTest.npz"))

print(f" Data loaded: target_train={len(target_train)}, target_test={len(target_test)}")

# -------------------- HELPERS --------------------
def _free(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

def train_and_eval(defense: str, param: float, top_k: int = 3):
    """
    Train target + shadow with the same pipeline (except defense),
    then run ML-Leaks attack (top-k on logits).
    Returns dict with target acc and attack metrics.
    """
    print(f"\n{'='*70}\n{defense.upper()}  param={param}\n{'='*70}")

    kwargs = dict(
        dataset=(target_train, target_train_label, target_test, target_test_label),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        n_hidden=n_hidden,
        model=classifierType,
        trainer=defense,
        use_augmentation=USE_AUGMENTATION,
    )

    if defense == "mg":
        kwargs["mg_kappa"] = float(param)
    elif defense == "dp":
        kwargs["dp_noise_multiplier"] = float(param)
        kwargs["dp_max_grad_norm"] = float(DP_MAX_GRAD_NORM)
    else:
        raise ValueError("defense must be 'mg' or 'dp'")

    # --- Train TARGET ---
    print("Training TARGET...")
    tX, tY, tmodel = dp.train_target_model(**kwargs)

    # Evaluate TARGET
    tmodel.eval()
    device = next(tmodel.parameters()).device
    with torch.no_grad():
        preds = tmodel(torch.from_numpy(target_test).to(device).float()).argmax(1).cpu().numpy()
    acc = accuracy_score(target_test_label, preds)
    print(f"✓ Target accuracy: {acc*100:.2f}%")

    _free(tmodel)

    # --- Train SHADOW ---
    print("Training SHADOW...")
    kwargs["dataset"] = (shadow_train, shadow_train_label, shadow_test, shadow_test_label)
    sX, sY, smodel = dp.train_target_model(**kwargs)
    _free(smodel)

    # --- Attack ---
    print("Running ML-Leaks attack...")
    sXc = clip_top_k(sX, top=top_k)
    tXc = clip_top_k(tX, top=top_k)
    _, metrics = train_attack_classifier_and_eval(sXc, sY, tXc, tY, balance=True)

    result = dict(
        defense=defense,
        param=float(param),
        acc=float(acc),                    # target accuracy (0..1)
        prec=float(metrics["prec"]),
        rec=float(metrics["rec"]),
        auc=float(metrics.get("auc", np.nan)),
        top_k=int(top_k),
    )
    print(f"✓ Attack: prec={result['prec']:.3f}, rec={result['rec']:.3f}, auc={result['auc']:.3f}")

    _free(sX, sY, tX, tY, sXc, tXc)
    return result

def sort_by_acc(acc, *arrays):
    idx = np.argsort(acc)
    sorted_arrays = [acc[idx]]
    for a in arrays:
        sorted_arrays.append(a[idx])
    return sorted_arrays

# -------------------- RUN SWEEPS --------------------
mg_results = []
for kappa in mg_kappas:
    mg_results.append(train_and_eval("mg", kappa, top_k=3))
    _free()

dp_results = []
for sigma in dp_noise_multipliers:
    dp_results.append(train_and_eval("dp", sigma, top_k=3))
    _free()

# Save raw JSON
raw_json_path = f"{out_base}_raw_results.json"
with open(raw_json_path, "w") as f:
    json.dump({"mg": mg_results, "dp": dp_results}, f, indent=2)
print(" Saved:", raw_json_path)

# -------------------- ISO-ACCURACY COMPARISON --------------------
mg_acc  = np.array([r["acc"]*100 for r in mg_results])
mg_prec = np.array([r["prec"] for r in mg_results])
mg_rec  = np.array([r["rec"]  for r in mg_results])

dp_acc  = np.array([r["acc"]*100 for r in dp_results])
dp_prec = np.array([r["prec"] for r in dp_results])
dp_rec  = np.array([r["rec"]  for r in dp_results])

# sort for interpolation correctness
mg_acc, mg_prec, mg_rec = sort_by_acc(mg_acc, mg_prec, mg_rec)
dp_acc, dp_prec, dp_rec = sort_by_acc(dp_acc, dp_prec, dp_rec)

acc_min = max(mg_acc.min(), dp_acc.min())
acc_max = min(mg_acc.max(), dp_acc.max())

print("\nAccuracy ranges:")
print(f"  MG: [{mg_acc.min():.1f}%, {mg_acc.max():.1f}%]")
print(f"  DP: [{dp_acc.min():.1f}%, {dp_acc.max():.1f}%]")
print(f"  Overlap: [{acc_min:.1f}%, {acc_max:.1f}%]")

have_overlap = (acc_max > acc_min)

if have_overlap:
    acc_grid = np.linspace(acc_min, acc_max, 25)
    mg_prec_interp = np.interp(acc_grid, mg_acc, mg_prec)
    mg_rec_interp  = np.interp(acc_grid, mg_acc, mg_rec)
    dp_prec_interp = np.interp(acc_grid, dp_acc, dp_prec)
    dp_rec_interp  = np.interp(acc_grid, dp_acc, dp_rec)

    # Differences (DP - MG): positive means DP has HIGHER attack success (worse privacy)
    delta_prec = dp_prec_interp - mg_prec_interp
    delta_rec  = dp_rec_interp  - mg_rec_interp

    print("\nISO-ACCURACY TABLE (lower attack metric = better privacy)")
    print(f"{'Acc%':>7} | {'MG_prec':>8} | {'DP_prec':>8} | {'DP-MG':>8} || {'MG_rec':>7} | {'DP_rec':>7} | {'DP-MG':>7}")
    print("-"*80)
    for i in range(0, len(acc_grid), 4):
        print(f"{acc_grid[i]:7.2f} | {mg_prec_interp[i]:8.3f} | {dp_prec_interp[i]:8.3f} | {delta_prec[i]:8.3f} || "
              f"{mg_rec_interp[i]:7.3f} | {dp_rec_interp[i]:7.3f} | {delta_rec[i]:7.3f}")

    print("\nSummary across overlap:")
    print(f"  mean(DP-MG) precision: {delta_prec.mean():+.4f}")
    print(f"  mean(DP-MG) recall:    {delta_rec.mean():+.4f}")
else:
    acc_grid = None
    mg_prec_interp = mg_rec_interp = dp_prec_interp = dp_rec_interp = None
    delta_prec = delta_rec = None
    print("\n No overlap. Increase MG noise sweep and/or DP noise sweep to create overlap.")

# -------------------- PLOTS --------------------
os.makedirs(os.path.dirname(out_base), exist_ok=True)

# Plot 1: tradeoff curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.plot(mg_acc, mg_prec, "o-", linewidth=2.5, markersize=7, label="MG")
ax1.plot(dp_acc, dp_prec, "s--", linewidth=2.5, markersize=7, label="DP")
ax1.set_xlabel("Target Accuracy (%)")
ax1.set_ylabel("Attack Precision")
ax1.set_title("Accuracy vs Attack Precision")
ax1.grid(True, alpha=0.3, linestyle="--")
ax1.legend()

ax2.plot(mg_acc, mg_rec, "o-", linewidth=2.5, markersize=7, label="MG")
ax2.plot(dp_acc, dp_rec, "s--", linewidth=2.5, markersize=7, label="DP")
ax2.set_xlabel("Target Accuracy (%)")
ax2.set_ylabel("Attack Recall")
ax2.set_title("Accuracy vs Attack Recall")
ax2.grid(True, alpha=0.3, linestyle="--")
ax2.legend()

plt.tight_layout()
out_tradeoff = f"{out_base}_tradeoff.png"
plt.savefig(out_tradeoff, dpi=300, bbox_inches="tight")
plt.show()
print(" Saved:", out_tradeoff)

# Plot 2: iso-accuracy
if have_overlap:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

    ax1.plot(acc_grid, mg_prec_interp, "o-", linewidth=2.5, markersize=6, label="MG")
    ax1.plot(acc_grid, dp_prec_interp, "s--", linewidth=2.5, markersize=6, label="DP")
    ax1.set_xlabel("Target Accuracy (%)")
    ax1.set_ylabel("Attack Precision (iso-acc)")
    ax1.set_title("Iso-Accuracy: Precision")
    ax1.grid(True, alpha=0.3, linestyle="--")
    ax1.legend()

    ax2.plot(acc_grid, mg_rec_interp, "o-", linewidth=2.5, markersize=6, label="MG")
    ax2.plot(acc_grid, dp_rec_interp, "s--", linewidth=2.5, markersize=6, label="DP")
    ax2.set_xlabel("Target Accuracy (%)")
    ax2.set_ylabel("Attack Recall (iso-acc)")
    ax2.set_title("Iso-Accuracy: Recall")
    ax2.grid(True, alpha=0.3, linestyle="--")
    ax2.legend()

    plt.tight_layout()
    out_iso = f"{out_base}_iso_accuracy.png"
    plt.savefig(out_iso, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_iso)

    # Plot 3: difference
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.axhline(0.0, linewidth=1, alpha=0.4)
    ax.plot(acc_grid, delta_prec, "o-", linewidth=2.5, markersize=6, label="Δ Precision (DP - MG)")
    ax.plot(acc_grid, delta_rec,  "s--", linewidth=2.5, markersize=6, label="Δ Recall (DP - MG)")
    ax.set_xlabel("Target Accuracy (%)")
    ax.set_ylabel("Δ Attack Success (DP - MG)")
    ax.set_title("Positive = DP worse privacy, Negative = MG worse privacy")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.legend()

    plt.tight_layout()
    out_diff = f"{out_base}_difference.png"
    plt.savefig(out_diff, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_diff)

print("\nDONE.")
print("Raw JSON:", raw_json_path)
print("Plots   :", out_tradeoff)
if have_overlap:
    print("          ", out_iso)
    print("          ", out_diff)
